# ResNet18 Deployment & Flask Chatbot
## Plastic Waste Classification System

In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import json
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## 1. Load Model

In [2]:
MODEL_PATH = 'best_model_ResNet18.pth'
if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    num_classes = checkpoint['num_classes']
    idx_to_class = checkpoint['idx_to_class']
    model = models.resnet18(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    print(f"✅ Model loaded: {checkpoint['best_acc']:.4f} accuracy")
else:
    print('⚠️ Run 02_plastic_model_training.ipynb first')

✅ Model loaded: 0.9953 accuracy


c:\Users\vimal\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\vimal\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


## 2. Plastic Database

In [3]:
plastic_db = {
    'PET': {'full_name': 'Polyethylene Terephthalate', 'recycling_code': '#1 PET', 'recyclability': 'Highly Recyclable', 'color': 'green'},
    'HDPE': {'full_name': 'High-Density Polyethylene', 'recycling_code': '#2 HDPE', 'recyclability': 'Highly Recyclable', 'color': 'green'},
    'PVC': {'full_name': 'Polyvinyl Chloride', 'recycling_code': '#3 PVC', 'recyclability': 'Rarely Recyclable', 'color': 'red'},
    'LDPE': {'full_name': 'Low-Density Polyethylene', 'recycling_code': '#4 LDPE', 'recyclability': 'Sometimes Recyclable', 'color': 'orange'},
    'PP': {'full_name': 'Polypropylene', 'recycling_code': '#5 PP', 'recyclability': 'Recyclable', 'color': 'blue'},
    'PS': {'full_name': 'Polystyrene', 'recycling_code': '#6 PS', 'recyclability': 'Rarely Recyclable', 'color': 'red'},
    'OTHERS': {'full_name': 'Other Plastics', 'recycling_code': '#7 OTHER', 'recyclability': 'Check Guidelines', 'color': 'gray'}
}
print('Database ready')

Database ready


## 3. Create Flask App

In [4]:
flask_code = '''from flask import Flask, request, jsonify, render_template
from flask_cors import CORS
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import json

app = Flask(__name__)
CORS(app)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('best_model_ResNet18.pth', map_location=device)
num_classes = checkpoint['num_classes']
idx_to_class = checkpoint['idx_to_class']

model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

plastic_db = ''' + json.dumps(plastic_db) + '''

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/health')
def health():
    return jsonify({'status': 'healthy'})

@app.route('/classify', methods=['POST'])
def classify():
    try:
        file = request.files['file']
        image = Image.open(file.stream).convert('RGB')
        img_tensor = transform(image).unsqueeze(0).to(device)
        
        with torch.no_grad():
            outputs = model(img_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            confidence, predicted = torch.max(probs, 1)
        
        # Convert idx_to_class keys to integers if they're strings
        if isinstance(list(idx_to_class.keys())[0], str):
            idx_to_class_int = {int(k): v for k, v in idx_to_class.items()}
        else:
            idx_to_class_int = idx_to_class
        
        predicted_idx = predicted.item()
        plastic_type = idx_to_class_int.get(predicted_idx, 'UNKNOWN')
        
        return jsonify({
            'plastic_type': plastic_type,
            'confidence': float(confidence.item()),
            'info': plastic_db.get(plastic_type, {}),
            'success': True
        })
    except Exception as e:
        print(f"Error in classify: {e}")
        return jsonify({'error': str(e), 'success': False}), 500

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)
'''

with open('app.py', 'w') as f:
    f.write(flask_code)
print('✅ app.py created')

✅ app.py created


## 4. Create HTML Interface

In [5]:
os.makedirs('templates', exist_ok=True)

# Create complete HTML file with enhanced display
html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Plastic Waste Classifier</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }
        .container {
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            max-width: 800px;
            width: 100%;
            padding: 40px;
        }
        h1 {
            text-align: center;
            color: #333;
            margin-bottom: 10px;
            font-size: 2.5em;
        }
        .subtitle {
            text-align: center;
            color: #666;
            margin-bottom: 30px;
        }
        .upload-area {
            border: 3px dashed #667eea;
            border-radius: 15px;
            padding: 60px 20px;
            text-align: center;
            cursor: pointer;
            transition: all 0.3s;
            margin-bottom: 20px;
        }
        .upload-area:hover {
            background: #f8f9ff;
            border-color: #764ba2;
        }
        .upload-icon {
            font-size: 60px;
            margin-bottom: 20px;
        }
        .upload-text {
            font-size: 18px;
            color: #666;
        }
        #fileInput { display: none; }
        .preview-container {
            text-align: center;
            margin: 20px 0;
            display: none;
        }
        #imagePreview {
            max-width: 100%;
            max-height: 300px;
            border-radius: 10px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.2);
        }
        .btn {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            padding: 15px 40px;
            font-size: 16px;
            border-radius: 50px;
            cursor: pointer;
            transition: transform 0.2s;
            display: block;
            margin: 20px auto;
        }
        .btn:hover { transform: scale(1.05); }
        .btn:disabled {
            opacity: 0.6;
            cursor: not-allowed;
        }
        .loading {
            display: none;
            text-align: center;
            margin: 20px 0;
        }
        .spinner {
            border: 4px solid #f3f3f3;
            border-top: 4px solid #667eea;
            border-radius: 50%;
            width: 50px;
            height: 50px;
            animation: spin 1s linear infinite;
            margin: 0 auto;
        }
        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }
        .result {
            display: none;
            margin-top: 30px;
            padding: 30px;
            border-radius: 15px;
            animation: fadeIn 0.5s;
        }
        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(20px); }
            to { opacity: 1; transform: translateY(0); }
        }
        .result.green { background: #d4edda; border-left: 5px solid #28a745; }
        .result.red { background: #f8d7da; border-left: 5px solid #dc3545; }
        .result.orange { background: #fff3cd; border-left: 5px solid #ffc107; }
        .result.blue { background: #d1ecf1; border-left: 5px solid #17a2b8; }
        .result.gray { background: #e2e3e5; border-left: 5px solid #6c757d; }
        .result-header {
            font-size: 32px;
            font-weight: bold;
            margin-bottom: 10px;
        }
        .confidence {
            font-size: 20px;
            margin-bottom: 20px;
            color: #555;
        }
        .info-item {
            margin: 10px 0;
            padding: 10px;
            background: rgba(255,255,255,0.5);
            border-radius: 8px;
        }
        .info-label {
            font-weight: bold;
            color: #333;
        }
        .info-value {
            color: #666;
            margin-top: 5px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>♻️ Plastic Waste Classifier</h1>
        <p class="subtitle">AI-Powered Recycling Assistant</p>
        
        <div class="upload-area" id="uploadArea">
            <div class="upload-icon">📸</div>
            <div class="upload-text">Drag & Drop or Click to Upload</div>
            <div class="upload-text" style="font-size: 14px; margin-top: 10px;">Supports: JPG, PNG</div>
        </div>
        
        <input type="file" id="fileInput" accept="image/*">
        
        <div class="preview-container" id="previewContainer">
            <img id="imagePreview" alt="Preview">
        </div>
        
        <button class="btn" id="classifyBtn" style="display:none;">Classify Plastic</button>
        
        <div class="loading" id="loading">
            <div class="spinner"></div>
            <p style="margin-top: 15px; color: #666;">Analyzing image...</p>
        </div>
        
        <div class="result" id="result"></div>
    </div>

    <script>
        const uploadArea = document.getElementById('uploadArea');
        const fileInput = document.getElementById('fileInput');
        const previewContainer = document.getElementById('previewContainer');
        const imagePreview = document.getElementById('imagePreview');
        const classifyBtn = document.getElementById('classifyBtn');
        const loading = document.getElementById('loading');
        const result = document.getElementById('result');
        let selectedFile = null;

        uploadArea.addEventListener('click', () => fileInput.click());
        
        uploadArea.addEventListener('dragover', (e) => {
            e.preventDefault();
            uploadArea.style.background = '#f8f9ff';
        });
        
        uploadArea.addEventListener('dragleave', () => {
            uploadArea.style.background = '';
        });
        
        uploadArea.addEventListener('drop', (e) => {
            e.preventDefault();
            uploadArea.style.background = '';
            const files = e.dataTransfer.files;
            if (files.length > 0) handleFile(files[0]);
        });
        
        fileInput.addEventListener('change', (e) => {
            if (e.target.files.length > 0) handleFile(e.target.files[0]);
        });
        
        function handleFile(file) {
            if (!file.type.startsWith('image/')) {
                alert('Please upload an image file');
                return;
            }
            
            selectedFile = file;
            const reader = new FileReader();
            reader.onload = (e) => {
                imagePreview.src = e.target.result;
                previewContainer.style.display = 'block';
                classifyBtn.style.display = 'block';
                result.style.display = 'none';
            };
            reader.readAsDataURL(file);
        }
        
        classifyBtn.addEventListener('click', async () => {
            if (!selectedFile) return;
            
            classifyBtn.disabled = true;
            loading.style.display = 'block';
            result.style.display = 'none';
            
            const formData = new FormData();
            formData.append('file', selectedFile);
            
            try {
                const response = await fetch('/classify', {
                    method: 'POST',
                    body: formData
                });
                const data = await response.json();
                
                if (data.success) {
                    const info = data.info;
                    result.className = 'result ' + info.color;
                    
                    // Determine recyclability icon and message
                    let recycleIcon = '';
                    let recycleMessage = '';
                    if (info.color === 'green') {
                        recycleIcon = '♻️✅';
                        recycleMessage = 'YES - This plastic is highly recyclable!';
                    } else if (info.color === 'blue') {
                        recycleIcon = '♻️';
                        recycleMessage = 'YES - This plastic is recyclable!';
                    } else if (info.color === 'orange') {
                        recycleIcon = '⚠️';
                        recycleMessage = 'SOMETIMES - Check with your local recycling facility';
                    } else if (info.color === 'red') {
                        recycleIcon = '❌';
                        recycleMessage = 'NO - This plastic is rarely recyclable';
                    } else {
                        recycleIcon = '❓';
                        recycleMessage = 'UNKNOWN - Check local guidelines';
                    }
                    
                    result.innerHTML = `
                        <div class="result-header">${info.recycling_code} ${data.plastic_type}</div>
                        <div class="confidence">Confidence: ${(data.confidence * 100).toFixed(1)}%</div>
                        
                        <div class="info-item" style="background: rgba(255,255,255,0.8); padding: 15px; margin: 15px 0;">
                            <div class="info-label" style="font-size: 18px; color: #333;">📝 Full Form:</div>
                            <div class="info-value" style="font-size: 20px; font-weight: bold; color: #000; margin-top: 8px;">${info.full_name}</div>
                        </div>
                        
                        <div class="info-item" style="background: rgba(255,255,255,0.8); padding: 15px; margin: 15px 0;">
                            <div class="info-label" style="font-size: 18px; color: #333;">${recycleIcon} Recyclable?</div>
                            <div class="info-value" style="font-size: 20px; font-weight: bold; color: #000; margin-top: 8px;">${recycleMessage}</div>
                        </div>
                        
                        <div class="info-item" style="background: rgba(255,255,255,0.8); padding: 15px; margin: 15px 0;">
                            <div class="info-label" style="font-size: 16px; color: #333;">ℹ️ Status:</div>
                            <div class="info-value" style="font-size: 18px; margin-top: 8px;">${info.recyclability}</div>
                        </div>
                    `;
                    result.style.display = 'block';
                } else {
                    alert('Error: ' + data.error);
                }
            } catch (error) {
                alert('Error: ' + error.message);
            } finally {
                loading.style.display = 'none';
                classifyBtn.disabled = false;
            }
        });
    </script>
</body>
</html>
"""

with open('templates/index.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print('✅ templates/index.html created with enhanced display')

✅ templates/index.html created with enhanced display


## 5. Create Requirements

In [6]:
requirements = '''flask==2.3.0
flask-cors==4.0.0
torch==2.0.0
torchvision==0.15.0
Pillow==10.0.0
numpy==1.24.0
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements)
print('✅ requirements.txt created')

✅ requirements.txt created


## 6. Create README

In [7]:
readme = '''# Plastic Waste Classifier - Flask Chatbot

## Setup
```bash
pip install -r requirements.txt
```

## Run
```bash
python app.py
```

## Access
Open browser: http://localhost:5000

## Usage
1. Upload plastic waste image
2. Click "Classify Plastic"
3. View results and recycling info
'''

with open('README.md', 'w') as f:
    f.write(readme)
print('✅ README.md created')

✅ README.md created


## 7. Summary

In [8]:
print('='*60)
print('DEPLOYMENT PACKAGE CREATED')
print('='*60)
print('Files created:')
print('  ✅ app.py - Flask backend')
print('  ✅ templates/index.html - Web interface')
print('  ✅ requirements.txt - Dependencies')
print('  ✅ README.md - Instructions')
print('\nTo run:')
print('  1. pip install -r requirements.txt')
print('  2. python app.py')
print('  3. Open http://localhost:5000')
print('='*60)

DEPLOYMENT PACKAGE CREATED
Files created:
  ✅ app.py - Flask backend
  ✅ templates/index.html - Web interface
  ✅ requirements.txt - Dependencies
  ✅ README.md - Instructions

To run:
  1. pip install -r requirements.txt
  2. python app.py
  3. Open http://localhost:5000
